# RAG + RAGAS — Walkthrough Interativo

Este notebook guia você pelo processo completo:
1. Montar o pipeline RAG
2. Gerar o Golden Dataset automaticamente
3. Avaliar com métricas RAGAS
4. Interpretar os resultados


In [ ]:
# Setup inicial
import os, sys
from pathlib import Path

# Adiciona o src ao path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

# Configure sua chave aqui ou use um arquivo .env
os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'   # ← substitua pela sua chave

print('✅ Setup OK')

---
## Parte 1 — Pipeline RAG

Monta o pipeline completo: carrega documentos → cria chunks → embeddings → ChromaDB.

In [ ]:
from rag_pipeline import RAGPipeline

rag = RAGPipeline()
rag.build()  # processa os documentos de data/sample_docs/

In [ ]:
# Teste rápido — faça uma pergunta
resultado = rag.query('Quando foi cunhado o termo inteligência artificial?')

print(f"❓ Pergunta: {resultado['question']}")
print(f"💬 Resposta: {resultado['answer']}")
print(f"\n📄 Contextos recuperados ({len(resultado['contexts'])}):'")
for i, ctx in enumerate(resultado['contexts'], 1):
    print(f"  [{i}] {ctx[:200]}...")

---
## Parte 2 — Geração do Golden Dataset com RAGAS

O RAGAS analisa os documentos e gera **perguntas + ground truth** automaticamente.

Isso resolve o maior desafio na avaliação de RAG: criar o dataset sem trabalho manual.

```
Seus documentos
      │
      ▼
  RAGAS TestsetGenerator
      │
      ▼
  Question + Ground Truth  ← golden dataset!
```

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from ragas.testset.generator import TestsetGenerator
from ragas.testset.evolutions import simple, reasoning, multi_context
from langchain_anthropic import ChatAnthropic, HuggingFaceEmbeddings

# Carrega documentos
DOCS_DIR = Path.cwd().parent / 'data' / 'sample_docs'
loader = DirectoryLoader(str(DOCS_DIR), glob='**/*.txt', loader_cls=TextLoader,
                         loader_kwargs={'encoding': 'utf-8'})
docs = loader.load()
print(f'📂 {len(docs)} documento(s) carregado(s)')

In [ ]:
# Configura o gerador RAGAS
generator = TestsetGenerator.from_langchain(
    generator_llm=ChatAnthropic(model='claude-haiku-4-5-20251001', temperature=0.3),
    critic_llm=ChatAnthropic(model='claude-sonnet-4-6', temperature=0),
    embeddings=HuggingFaceEmbeddings(model='text-embedding-3-small'),
)

# Gera 8 perguntas com distribuição variada
testset = generator.generate_with_langchain_docs(
    documents=docs,
    test_size=8,
    distributions={simple: 0.5, reasoning: 0.3, multi_context: 0.2},
)

df_golden = testset.to_pandas()
print(f'✅ {len(df_golden)} perguntas geradas!')
df_golden[['question', 'ground_truth', 'question_type']].head()

In [ ]:
# Inspeciona uma pergunta de cada tipo
for tipo in df_golden['question_type'].unique():
    row = df_golden[df_golden['question_type'] == tipo].iloc[0]
    print(f"\n🏷️  Tipo: {tipo}")
    print(f"   ❓ {row['question']}")
    print(f"   ✅ {row['ground_truth'][:200]}")

# Salva
output_path = Path.cwd().parent / 'outputs' / 'golden_dataset.csv'
output_path.parent.mkdir(exist_ok=True)
df_golden.to_csv(output_path, index=False)
print(f"\n💾 Salvo em: {output_path}")

---
## Parte 3 — Avaliação com Métricas RAGAS

Agora rodamos o RAG em cada pergunta do golden dataset e medimos:

| Métrica | Compara | Mede |
|---|---|---|
| **Faithfulness** | Answer ↔ Context | O LLM usou o contexto? |
| **Answer Correctness** | Answer ↔ Ground Truth | A resposta está certa? |
| **Answer Relevance** | Answer ↔ Question | A resposta é relevante? |
| **Context Precision** | Context ↔ Question+GT | O contexto é preciso? |
| **Context Recall** | Context ↔ Ground Truth | O contexto é completo? |

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_correctness, answer_relevancy, context_precision, context_recall

# Roda o RAG em cada pergunta do golden dataset
ragas_data = {'question': [], 'answer': [], 'contexts': [], 'ground_truth': []}

for i, row in df_golden.iterrows():
    result = rag.query(row['question'])
    ragas_data['question'].append(row['question'])
    ragas_data['answer'].append(result['answer'])
    ragas_data['contexts'].append(result['contexts'])
    ragas_data['ground_truth'].append(row['ground_truth'])
    print(f'  [{i+1}/{len(df_golden)}] ✓')

print('\n✅ RAG executado em todas as perguntas!')

In [ ]:
# Avalia com RAGAS
dataset = Dataset.from_dict(ragas_data)

result = evaluate(
    dataset=dataset,
    metrics=[faithfulness, answer_correctness, answer_relevancy, context_precision, context_recall]
)

df_scores = result.to_pandas()

# Scores médios
metric_cols = ['faithfulness', 'answer_correctness', 'answer_relevancy', 'context_precision', 'context_recall']
print('\n📊 SCORES MÉDIOS:')
print(df_scores[metric_cols].mean().round(3).to_string())

In [ ]:
# Visualiza scores por pergunta
df_scores[['question', 'faithfulness', 'answer_correctness', 'answer_relevancy']].round(3)

In [ ]:
# 🔍 Insight importante:
# Faithfulness baixa + Answer Correctness alta =
#   LLM usando conhecimento próprio, não o contexto recuperado!

problematic = df_scores[
    (df_scores['faithfulness'] < 0.5) & 
    (df_scores['answer_correctness'] > 0.6)
]

if len(problematic) > 0:
    print('⚠️  Perguntas onde o LLM usou conhecimento próprio (não o contexto):')
    for _, row in problematic.iterrows():
        print(f"   → {row['question']}")
        print(f"     faithfulness={row['faithfulness']:.2f}, correctness={row['answer_correctness']:.2f}")
else:
    print('✅ Todas as respostas corretas foram baseadas no contexto recuperado!')

# Salva resultados finais
scores_path = Path.cwd().parent / 'outputs' / 'evaluation_scores.csv'
df_scores.to_csv(scores_path, index=False)
print(f'\n💾 Scores salvos em: {scores_path}')